# Exploratory Data Analysis (EDA)

This notebook focuses on extracting practical forecasting insights from energy consumption data.

## 1. Data Overview (Sanity Check)

### Objective
Confirm that the dataset is structurally valid before any deeper analysis.

### What to inspect
- Dataset shape
- Data types
- Percentage of missing values

### Expected conclusions
- Data structure is consistent
- Missing values are within expected limits
- No obvious structural corruption

A weak data overview invalidates all downstream conclusions.

In [17]:
# Ensure project root is importable even when running from notebooks/
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_data

path = project_root / "data" / "raw" / "data.txt"
data = load_data(path)
data.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## 2. Time Series Visualization (Multi-Scale View)

### Objective
Understand behavior at global, medium, and short-term time scales.

### Required views
1. Full timeline (sampled) to inspect long-term trend
2. One-month window for medium-term variation
3. One-week window for daily cycle visibility

### What to extract
- Presence and stability of daily patterns
- Peak consistency over time
- Unusual periods or anomalies

## 3. Seasonality and Trend Decomposition

### Objective
Separate the signal into interpretable components.

### Components to analyze
- Trend
- Seasonal
- Residual

### Interpretation guide
- Strong seasonal component suggests SARIMA may perform well
- Large/noisy residuals may benefit from ML-based models
- A visible trend suggests differencing may be required

## 4. Autocorrelation Analysis (Feature Direction)

### Objective
Identify time dependencies that should guide lag-feature design.

### Plots to run
- ACF
- PACF

### Key interpretations
- Significant spike at lag 24 implies daily dependency
- Significant spike at lag 168 implies weekly dependency

### Modeling implication
Use significant lags as candidate predictive features rather than guessing.

## 5. Distribution Analysis

### Objective
Understand target distribution shape and modeling risk.

### Plots and checks
- Histogram
- Optional log-transform diagnostics

### What to determine
- Degree of skewness
- Presence of extreme values

### Modeling implication
- Strong skewness can reduce model stability
- Heavy tails can justify anomaly-focused analysis

## 6. Correlation Analysis

### Objective
Assess whether variables add unique signal or redundant information.

### Variables to compare
- Global_active_power
- Voltage
- Global_intensity
- Sub_metering variables

### Interpretation guide
- Strong correlation may indicate redundancy
- Weak/moderate correlation can indicate complementary signal

### Example decision
If Global_intensity is highly collinear with the target, it may add limited incremental value.

## 7. Sub-Metering Deep Dive

### Objective
Convert generic forecasting into interpretable energy-behavior analysis.

### Analyses to perform
- Average usage by sub-meter
- Time-of-day consumption profiles
- Seasonal variation by sub-meter

### Questions to answer
- Are kitchen-related loads peaking at expected hours?
- Is climate-related usage seasonally elevated?
- Are laundry/cyclic appliance patterns detectable?

## 8. Missing Data Pattern Analysis

### Objective
Validate that missing-value handling did not hide systematic issues.

### What to visualize
- Missingness over time

### Interpretation guide
- Random isolated gaps are usually acceptable
- Clustered gaps can indicate sensor/system outages

## 9. Key Insights

### 9.1 Daily Consumption Cycle
Energy consumption shows a strong daily cycle, with repeatable peak hours.

### 9.2 Weekly Seasonality
Weekly seasonality is visible, with distinct weekday/weekend behavior.

### 9.3 Predictive Lag Structure
Autocorrelation supports lag 24 and lag 168 as high-value predictive lags.

### 9.4 Sub-Metering Seasonal Effects
Sub_metering_3 exhibits seasonal spikes consistent with climate-related demand.

### 9.5 Distribution Risk Signal
The target distribution is right-skewed, indicating occasional high-consumption events.